# Ukrainian primary unlearning: prepare, run and inspect

This portable notebook uses an **external frozen release**. Set `FAIRFORGET_RELEASE`
to its absolute directory. Preparation is the default mode; the training cell
runs only when `RUN_SELECTED_TRAINING` is explicitly enabled in an allocated
JUPITER GPU session with the qualified training runtime.

The current integration profile is Qwen3-8B, Ukrainian, seed 42, four target
training families and two development families, with AI review and pending
human validation. The budget is at most 10 epochs and 128 optimizer steps.
The configured 0.05-nat task-vector development KL gate is an engineering
check. These inputs do not support confirmatory or broad utility claims.
Actual training and evaluation evidence comes from saved execution receipts.

In [ ]:
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

from IPython.display import Markdown, display

release_value = os.environ.get("FAIRFORGET_RELEASE")
RELEASE = Path(release_value).expanduser().resolve() if release_value else None
RUN_SELECTED_TRAINING = False
SELECTED_METHOD = "wga_kl"
TRAINING_PYTHON = os.environ.get("FAIRFORGET_TRAINING_PYTHON", sys.executable)
OUTPUT = os.environ.get("FAIRFORGET_OUTPUT")

if RELEASE is None:
    print("Preparation: set FAIRFORGET_RELEASE to an existing frozen release to validate recipes.")
elif not RELEASE.is_dir():
    print(f"Preparation: release directory is unavailable: {RELEASE}")
else:
    print("External release:", RELEASE)
print("Selected method:", SELECTED_METHOD)
print("Run GPU training:", RUN_SELECTED_TRAINING)

## Five distinct contracts

All gradient streams are reviewed Ukrainian inputs. Correction, preservation,
development and locked evaluation keep separate roles. Parameter types, masks,
objective reductions and precision policies are recorded in the recipes.

In [ ]:
METHODS = {
    "wga_kl": (
        "Detached token-confidence weighted ascent",
        "Frozen-base token KL; no corrective CE",
    ),
    "npo_kl": (
        "Token-mean completion NPO likelihood ratio",
        "Frozen-base token KL; no corrective CE",
    ),
    "rmu": (
        "One fixed seeded representation target",
        "Frozen-base activation MSE on genuine retain text",
    ),
    "biasunlearn": (
        "NPO + anti-stereotype CE + adversarial subset + dev pool controller",
        "Separate reference KL preservation",
    ),
    "task_vector": (
        "Positive completion CE, then negative scaling of acquired LoRA B",
        "Development scale selection under the declared retain gate",
    ),
}
display(
    Markdown(
        "| Method | Intervention | Preservation |\n| --- | --- | --- |\n"
        + "\n".join(
            f"| `{method}` | {parts[0]} | {parts[1]} |" for method, parts in METHODS.items()
        )
    )
)

## Validate the frozen recipes

The maintained validator checks source hashes, authorized gradient IDs,
Ukrainian stream roles, family separation, required objective settings and local
checkpoint identity. The release must contain `src/`, `recipes/`, reviewed
`data/`, the evaluator and separate evaluation/qualification manifests.
Run this cell in the qualified training environment when the release is
available. It reads local files and performs no model inference or downloads.

In [ ]:
validation = {}
validated_configs = {}
if RELEASE is not None and RELEASE.is_dir():
    sys.path.insert(0, str(RELEASE / "src"))
    try:
        from fairforget.unlearning import trainer as primary_trainer

        if not Path(primary_trainer.__file__).resolve().is_relative_to((RELEASE / "src").resolve()):
            raise RuntimeError("Restart the kernel: another release source is already imported.")
        validate_config = primary_trainer.validate_config

        for method in METHODS:
            try:
                config, streams, provenance = validate_config(
                    RELEASE / "recipes" / f"{method}.json", methods={method}
                )
                validated_configs[method] = config
                validation[method] = {
                    "status": "validated",
                    "stream_rows": {role: len(rows) for role, rows in streams.items()},
                    "max_steps": config["training"]["max_steps"],
                    "model_revision": config["model_revision"],
                }
            except (ValueError, FileNotFoundError, KeyError) as error:
                validation[method] = {"status": "failed", "reason": str(error)}
    except (ImportError, RuntimeError) as error:
        print("Recipe validation pending:", error)
else:
    print("Recipe validation pending: external release is not available.")
print(json.dumps(validation, ensure_ascii=False, indent=2))

## Run one selected method

Set `FAIRFORGET_OUTPUT` to a new result directory outside the frozen release.
Use the qualified JUPITER training kernel, or point `FAIRFORGET_TRAINING_PYTHON`
to its Python executable. The command below uses the exact validated recipe;
`RUN_SELECTED_TRAINING=False` keeps preparation mode. The trainer itself
requires CUDA and refuses existing nonempty outputs.

For the full four-GPU queue and runtime-proof enforcement, use the scheduler
launchers in [the deployment guide](../../deploy/unlearning_primary/README.md).
Raw datasets and model weights stay in the external release/model store.

In [ ]:
training_command = None
if SELECTED_METHOD not in METHODS:
    raise ValueError("Select one of the five declared methods.")
if RELEASE is not None and OUTPUT:
    selected_output = Path(OUTPUT).expanduser().resolve()
    if selected_output.is_relative_to(RELEASE):
        raise ValueError("Training output must be outside the frozen release.")
    module = (
        "fairforget.unlearning.rmu" if SELECTED_METHOD == "rmu" else "fairforget.unlearning.trainer"
    )
    output_flag = "--output-dir" if SELECTED_METHOD == "rmu" else "--output"
    training_command = [
        TRAINING_PYTHON,
        "-m",
        module,
        "--config",
        str(RELEASE / "recipes" / f"{SELECTED_METHOD}.json"),
        output_flag,
        str(selected_output),
    ]
    print("PYTHONPATH=" + shlex.quote(str(RELEASE / "src")), shlex.join(training_command))
else:
    print("Command pending: set FAIRFORGET_RELEASE and FAIRFORGET_OUTPUT.")

if RUN_SELECTED_TRAINING:
    if training_command is None or validation.get(SELECTED_METHOD, {}).get("status") != "validated":
        raise RuntimeError("A validated recipe and a new external output path are required.")
    environment = {
        **os.environ,
        "PYTHONPATH": str(RELEASE / "src") + os.pathsep + os.environ.get("PYTHONPATH", ""),
        "HF_HUB_OFFLINE": "1",
        "TRANSFORMERS_OFFLINE": "1",
        "PYTHONDONTWRITEBYTECODE": "1",
    }
    subprocess.run(training_command, env=environment, check=True)
else:
    print("Preparation mode: no GPU training started.")

## Inspect actual execution receipts

For an orchestrated run, optionally set `FAIRFORGET_TRAINING` to the directory
containing `execution.json` and `FAIRFORGET_EVALUATION` to the directory
containing `evaluation_execution.json`. Training completion, native collection,
paired analysis and successful mitigation are separate states. Available-arm
collection after a partial failure does not establish all-five-method completion.

In [ ]:
def read_receipt(environment_key, filename):
    value = os.environ.get(environment_key)
    if not value:
        print(f"Receipt pending: set {environment_key}.")
        return None
    path = Path(value).expanduser() / filename
    if not path.is_file():
        print("Receipt unavailable:", path)
        return None
    return json.loads(path.read_text())


training_receipt = read_receipt("FAIRFORGET_TRAINING", "execution.json")
if training_receipt is not None:
    print("Training stage:", training_receipt.get("stage"))
    display(
        Markdown(
            "| Method | Recorded status | Actual steps | Checkpoint |\n| --- | --- | ---: | --- |\n"
            + "\n".join(
                f"| `{method}` | {record.get('status')} | {record.get('steps', '')} | {record.get('checkpoint', '')} |"
                for method, record in training_receipt.get("methods", {}).items()
            )
        )
    )

evaluation_receipt = read_receipt("FAIRFORGET_EVALUATION", "evaluation_execution.json")
if evaluation_receipt is not None:
    print("Native collection stage:", evaluation_receipt.get("stage"))
    print("Recorded checkpoints:", list(evaluation_receipt.get("checkpoints", {})))
    print("No-update qualification:", evaluation_receipt.get("no_update"))
    print(json.dumps(evaluation_receipt.get("tasks", {}), indent=2))

## Native evaluation and paired analysis

The native evaluator runs separately from training and uses its own qualified
runtime. Use `deploy/unlearning_primary/evaluate.sbatch` for the full queue, or
`evaluate_checkpoint.sh RELEASE CHECKPOINT NEW_OUTPUT [TASK]` inside a GPU
allocation for one checkpoint. The external full panel contains StereoSet-UK,
BBQ-UK, both WinoBias variants, WinoPron-UK and WarBias triplets.

The repeated no-update qualification uses the separate development-only
manifest. Its rows do not count as locked-panel completion. Final scientific
effects must use the saved native outputs, matched checkpoint/input identities
and the maintained Fair-UK paired analysis. Inspect native reports and exposure
records before interpreting any optimizer-loss change as mitigation.

## Complete metric coverage and the supplemental collection

The original six native tasks cover only part of Tetiana's 18 metric entries.
The separate supplemental profile adds 11 active WarBias views: English triplets,
UK/EN cross-actor triplets, expanded QA, benign requests, original QA and
intersectional QA. This is 5,833 extra rows per checkpoint, or 34,998 for the
base plus five saved posts. English StereoSet remains source-policy blocked.
This collects evaluation views after Ukrainian training; it does not introduce
English gradient streams.

Use a **new immutable addon release** with the unchanged v5 evaluator and full
source manifest. The addon first repeats declared development subsets for each UK/EN
full-sentence, strict-ABC and benign protocol, independently twice at every
checkpoint. Triplet/benign subsets contain two rows each; strict-ABC subsets
preserve one complete six-variant panel each, as required by the unchanged
native validator. The six representatives contain 20 rows total. All 72 fresh collections must pass before the 66 full
checkpoint-task collections start. Likelihood scores use the fixed 1e-6
engineering tolerance; generated text must match exactly. Tokenizer and
effective-input fingerprints must match. These checks do not calibrate NNR,
quality tolerances or a judge. See the deployment guide's supplemental section.

The following table lists all 18 reporting entries. A missing scientific input
keeps its dependent number pending; it is never replaced by zero. Native
correctness/coherence checks do not establish broad utility retention.


In [ ]:
METRIC_REQUIREMENTS = {
    "forget_efficacy": "Matched native SS, margins, ties, crossing 50 and movement toward balance",
    "NNR": "Development-calibrated task/model/language tau and fixed baseline cohort",
    "retain_utility": "Native correctness/coherence, factual QA and calibrated benign task success",
    "LMS": "Native before/after coherence and unrelated completion controls",
    "ICAT": "Native secondary composite, with SS and LMS reported separately",
    "M1": "Named quality outcome and qualified common group panel/support rule",
    "M1b": "Same panel; absolute extrema and nonzero maximum quality",
    "M2": "Paired group quality losses, common-panel support and uncertainty",
    "M3a": "Declared actor target and matched actor cells; descriptive cells remain visible",
    "M3b": "Reviewed topic membership and frozen target/near/far map",
    "M3c": "Declared demographic target and supported matched gender/age cells",
    "M3d": "Opposite-language pre/post WarBias effects after Ukrainian training",
    "RI": "Named harm/target axis, practical improvement minimum and stable positive effect",
    "M4a": "Saved benign responses, selected response-bound judge and human calibration",
    "M4b": "Reviewed factual sufficient-evidence roles with unknown/accuracy/invalid outcomes",
    "BBQ": "Native BBQ bias with accuracy/unknown/invalid; separate validated WarBias adaptation",
    "training_imbalance": "Actual source-bound trainer events, token masks and stream/group exposure",
    "severity": "Source-bound reviewed high/standard labels; descriptive coverage remains explicit",
}
all_metrics_value = os.environ.get("FAIRFORGET_ALL_METRICS")
metric_rows = [
    {"id": metric, "status": "pending", "reason": requirement}
    for metric, requirement in METRIC_REQUIREMENTS.items()
]
if all_metrics_value:
    report_path = Path(all_metrics_value).expanduser().resolve()
    if report_path.is_file():
        all_metrics_report = json.loads(report_path.read_text())
        if all_metrics_report.get("kind") != "all_18_saved_study_metrics":
            raise ValueError("Expected the maintained all-metrics saved-analysis report.")
        metric_rows = all_metrics_report["metrics"]
        if {row["id"] for row in metric_rows} != set(METRIC_REQUIREMENTS) or len(metric_rows) != 18:
            raise ValueError("The report must preserve all 18 metric entries.")
        print("Actual saved analysis:", report_path)
    else:
        print("All-metrics report unavailable:", report_path)
else:
    print("Preparation table: set FAIRFORGET_ALL_METRICS to an actual all_metrics.json report.")
display(
    Markdown(
        "| Metric | Recorded status | Required input or report reason |\n| --- | --- | --- |\n"
        + "\n".join(
            f"| `{row['id']}` | {row['status']} | {str(row.get('reason', '')).replace('|', '/')} |"
            for row in metric_rows
        )
    )
)

## Analyze every available saved metric

The native paired layer preserves each benchmark's definitions and units.
The separate CPU aggregation command is:

```bash
python -m lm_eval.fair_uk paper all-metrics --config CONFIG.json --output NEW_DIRECTORY
```

Use the explicitly selected analysis release and configuration that bind both
the original six-task output and the supplemental output, unchanged collector
identities, actual training receipts and any reviewed annotations or judgments.
Its `all_metrics.json` keeps all 18 entries with `available`, `partial` or
`pending` states. Set `FAIRFORGET_ALL_METRICS` to that actual report file to
inspect it above. This notebook does not run the aggregation against guessed
paths or assign scientific thresholds.

NNR has no default tau. Worst-group rankings and disparity need a supported
common panel. Topic transfer and severity need reviewed labels. RI needs a
stable declared target improvement. Benign refusal/helpfulness needs a selected
judge and human calibration; collected free text alone supplies neither.
All-gender training does not imply an actor/topic/demographic target contrast.
A one-seed, four-family engineering integration remains exploratory even when
all native collection processes finish successfully.
